# 01. Exploratory Data Analysis (EDA) & Data Cleaning
**Dự án:** Phân tích & Dự đoán Chất lượng Rượu Vang Đỏ (Red Wine Quality)
**Tập dữ liệu:** UCI Red Wine Quality Dataset (`winequality-red.csv`)

---
### Mục tiêu Notebook:
1. **Data Audit:** Kiểm tra cấu trúc dữ liệu, kiểu dữ liệu, thông tin tổng quan và giá trị khuyết thiếu (Missing Values).
2. **Outliers Analysis:** Phát hiện và đưa ra phương án xử lý các điểm ngoại lệ (Outliers).
3. **Exploratory Data Analysis (EDA):**
   - Phân tích đơn biến (Univariate Analysis) và đa biến (Multivariate Analysis).
   - Xác định các yếu tố hóa-lý có tương quan mạnh nhất đến chỉ số chất lượng (`quality`).
4. **Data Preparation:** Chuẩn hóa dữ liệu (Standardization) và chuẩn bị tập dữ liệu sạch cho bước Mô hình hóa.

In [ ]:
# 1. Import các thư viện cần thiết
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Cấu hình giao diện đồ thị
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

import warnings
warnings.filterwarnings('ignore')

print("Thư viện đã được tải thành công!")

## I. Khảo sát Dữ liệu (Data Audit)

Tải dữ liệu từ đường dẫn `data/raw/winequality-red.csv`.  
*Lưu ý: Tập dữ liệu UCI gốc thường dùng dấu chấm phẩy `;` làm delimiter.*

In [ ]:
# Load dataset
data_path = 'data/raw/winequality-red.csv'

# Tải dữ liệu (kiểm tra delimiter tự động hoặc chỉ định sep=';')
try:
    df = pd.read_csv(data_path, sep=';')
    if df.shape[1] == 1: # Nếu dùng sai sep, chỉ có 1 cột gom lại
        df = pd.read_csv(data_path, sep=',')
except Exception as e:
    print(f"Lỗi tải dữ liệu: {e}")

print(f"Kích thước tập dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột.\n")
df.head()

In [ ]:
# Thông tin tổng quan về các cột và kiểu dữ liệu
print("=== Thông tin dữ liệu (df.info()) ===")
df.info()

print("\n=== Kiểm tra giá trị khuyết thiếu (Missing Values) ===")
missing_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print(missing_summary)

### 📌 Kết luận về Missing Data:
- Tất cả các thuộc tính đều có giá trị missing = **0 (0%)**.
- **Cơ sở lý thuyết phân loại Missing Data:**
  - **MCAR** (Missing Completely at Random): Khuyết ngẫu nhiên hoàn toàn.
  - **MAR** (Missing at Random): Khuyết phụ thuộc thuộc tính khác.
  - **MNAR** (Missing Not at Random): Khuyết phụ thuộc vào chính giá trị bị khuyết.
- **Hành động:** Do tập dữ liệu hoàn toàn không bị khuyết, không cần thực hiện xóa dòng (`dropna`) hay điền giá trị khuyết (`imputation`).

## II. Thống kê Mô tả & Kiểm tra Outliers

Kiểm tra phân phối và khoảng vị trí (Min, Max, IQR) để phát hiện Outliers.

In [ ]:
# Thống kê mô tả
df.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]

In [ ]:
# Trực quan hóa Outliers bằng Boxplot cho các thuộc tính hóa học
features = df.columns[:-1] # Loại bỏ cột 'quality'

plt.figure(figsize=(15, 12))
for i, col in enumerate(features, 1):
    plt.subplot(4, 3, i)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(f'Boxplot: {col}')
    plt.tight_layout()

plt.show()

In [ ]:
# Hàm tính số lượng Outliers theo phương pháp IQR (Interquartile Range)
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print("=== Thống kê Outliers theo phương pháp IQR ===")
for col in features:
    count, lb, ub = detect_outliers_iqr(df, col)
    pct = (count / len(df)) * 100
    print(f"{col:22s}: {count:3d} outliers ({pct:5.2f}%) | Ngưỡng: [{lb:.2f}, {ub:.2f}]")

### 📌 Kết luận & Chiến lược Xử lý Outliers:
1. **Bản chất Outliers:** Các giá trị cực đại ở `residual sugar`, `chlorides`, `total sulfur dioxide` phản ánh tính chất hóa học thực tế của một số dòng rượu đặc biệt, không phải do lỗi nhập liệu.
2. **Quyết định:**
   - **Không xóa bỏ (No Truncation/Dropping):** Tránh giảm dung lượng tập dữ liệu nhỏ (1,599 mẫu) và làm mất thông tin cực đoan thực tế.
   - **Giới hạn giá trị (Capping / Winsorization):** Tiến hành co các giá trị vượt ngưỡng IQR về ranh giới верх/dưới để giảm độ lệch (skewness) đối với các mô hình tuyến tính.
   - **Lưu trữ phiên bản gốc:** Vẫn giữ nguyên tập dữ liệu chưa biến đổi cho các mô hình dạng Cây (Random Forest/XGBoost) vốn có khả năng chịu outlier tốt.

In [ ]:
# Kỹ thuật Capping Outliers (Winsorization thủ công bằng IQR)
df_capped = df.copy()

for col in features:
    Q1 = df_capped[col].quantile(0.25)
    Q3 = df_capped[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Clip giá trị trong phạm vi [lower_bound, upper_bound]
    df_capped[col] = np.clip(df_capped[col], lower_bound, upper_bound)

print("Đã thực hiện Capping Outliers thành công!")

## III. Phân tích Khám phá Dữ liệu (EDA) & Chẩn đoán

Phân tích mối quan hệ giữa biến mục tiêu `quality` và 11 chỉ số hóa học.

In [ ]:
# 1. Phân phối của biến mục tiêu 'quality'
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='quality', data=df, palette='viridis')
plt.title('Phân phối Điểm số Chất lượng Rượu Vang (Quality)', fontsize=14)
plt.xlabel('Điểm số Quality (3 - 8)')
plt.ylabel('Số lượng mẫu')

# Hiển thị số lượng cụ thể trên từng cột
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=10, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.show()

In [ ]:
# 2. Ma trận tương quan (Correlation Heatmap)
plt.figure(figsize=(12, 8))
correlation_matrix = df.corr()

sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Ma trận Tương quan Tương hỗ (Correlation Heatmap)', fontsize=14)
plt.savefig('images/correlation_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Top biến tương quan mạnh nhất tới 'quality'
corr_with_quality = correlation_matrix['quality'].drop('quality').sort_values(ascending=False)

plt.figure(figsize=(10, 5))
corr_with_quality.plot(kind='bar', color=['g' if x > 0 else 'r' for x in corr_with_quality])
plt.title('Mức độ tương quan của các chỉ số với Quality', fontsize=14)
plt.ylabel('Hệ số tương quan Pearson')
plt.axhline(0, color='black', linewidth=0.8)
plt.show()

In [ ]:
# 3. Phân tích tác động của các chỉ số hóa học chính lên Quality
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Alcohol vs Quality
sns.boxplot(x='quality', y='alcohol', data=df, ax=axes[0, 0], palette='Blues')
axes[0, 0].set_title('Alcohol vs Quality')

# Volatile Acidity vs Quality
sns.boxplot(x='quality', y='volatile acidity', data=df, ax=axes[0, 1], palette='Reds')
axes[0, 1].set_title('Volatile Acidity vs Quality')

# Sulphates vs Quality
sns.boxplot(x='quality', y='sulphates', data=df, ax=axes[1, 0], palette='Greens')
axes[1, 0].set_title('Sulphates vs Quality')

# Citric Acid vs Quality
sns.boxplot(x='quality', y='citric acid', data=df, ax=axes[1, 1], palette='Oranges')
axes[1, 1].set_title('Citric Acid vs Quality')

plt.tight_layout()
plt.show()

### 💡 Nhận xét từ Phân tích EDA (Diagnostic Analysis):
1. **Alcohol (Tương quan dương +0.48):** Rượu có nồng độ cồn cao hơn có xu hướng nhận được đánh giá chất lượng cao hơn rõ rệt.
2. **Volatile Acidity (Tương quan âm -0.39):** Axit bay hơi làm tăng hương vị giấm chua bẩn; hàm lượng này càng cao thì điểm số chất lượng rượu càng giảm.
3. **Sulphates (+0.25) & Citric Acid (+0.23):** Đóng vai trò phụ gia bảo quản và tạo vị tươi mát, có tương quan dương thuận lợi với chất lượng.
4. **Đa cộng tuyến (Multicollinearity):**
   - `fixed acidity` và `citric acid` có tương quan dương mạnh ($\approx +0.67$).
   - `fixed acidity` có tương quan âm rất mạnh với `pH` ($\approx -0.68$), đúng với bản chất hóa học (Axit tăng $\rightarrow$ pH giảm).

## IV. Chuẩn hóa & Biến đổi Dữ liệu (Data Processing)

Thiết lập biến mục tiêu bài toán Phân loại (Binary Classification):
- **1 (Good Wine):** `quality` $\ge 7$
- **0 (Average/Bad Wine):** `quality` $< 7$

In [ ]:
# Tạo nhãn phân loại nhị phân
df_processed = df_capped.copy()
df_processed['target'] = (df_processed['quality'] >= 7).astype(int)

print("Tỷ lệ phân phối lớp mục tiêu (Binary Target):")
print(df_processed['target'].value_counts(normalize=True) * 100)

# Chuẩn hóa thang đo các đặc trưng (Standardization)
X = df_processed.drop(columns=['quality', 'target'])
y = df_processed['target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Kiểm tra dữ liệu sau chuẩn hóa (Mean ~ 0, Std ~ 1)
X_scaled.describe().round(2).T[['mean', 'std']]

In [ ]:
# Lưu tập dữ liệu đã làm sạch & chuẩn hóa ra file CSV mới
output_dir = 'data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

cleaned_data_path = os.path.join(output_dir, 'winequality-red-cleaned.csv')
df_processed.to_csv(cleaned_data_path, index=False)

print(f"Đã lưu thành công tập dữ liệu sạch tại: {cleaned_data_path}")

---
### 🎯 Hoàn tất Notebook 01
Dữ liệu đã được làm sạch, xử lý outlier, phân tích tương quan và chuẩn bị sẵn sang cho giai đoạn Huấn luyện mô hình ở **`02_modeling_and_evaluation.ipynb`**.